# **Imports & Configuration**

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from urllib.parse import urlparse

# Scikit-Learn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# TensorFlow / Keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, BatchNormalization, Activation, MaxPooling1D, Dropout, GRU, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

#Evaluation Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

#(Hyperparameters)
MAX_LEN = 200          # Maximum link length
VOCAB_SIZE = 10000     # Number of distinctive letters/symbols
EMBEDDING_DIM = 128    # Dimensions of inclusion
BATCH_SIZE = 128
EPOCHS = 15
RANDOM_STATE = 42

# **Load & Clean Data**

In [ ]:
TRAIN_PATH = '/kaggle/input/deep-pro/deep_proj/Train (1).csv'
TEST_PATH = '/kaggle/input/deep-pro/deep_proj/Test (1).csv'
tranco_PATH = '/kaggle/input/tranco/tranco_PLYWJ.csv'

df = pd.read_csv(tranco_PATH, names=['rank', 'domain'])
tranco_dict = dict(zip(df['domain'], df['rank']))
print("tranco data load 'DONE' ")

df = pd.read_csv(TRAIN_PATH)

df.drop_duplicates(subset=['url'], inplace=True)
df = df.dropna(subset=['url', 'label'])

mapping = {'benign': 0, 'malicious': 1}

if df['label'].dtype == 'object':
    df['label'] = df['label'].replace(mapping)

df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"✅ Done : {len(df)}")
print(f"{df['label'].value_counts().to_dict()}")
df.head()

# **Domain-Aware Splitting**

### **1. Tokenization**

In [ ]:
!pip install pandarallel

In [ ]:
!pip install tldextract

In [ ]:
import string
import re
import math
import numpy as np
from collections import Counter
from urllib.parse import urlparse
from sklearn.preprocessing import StandardScaler
import tldextract
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

class URLPreprocessor:
    def __init__(self, tranco_dict, max_len=200):
        self.max_len = max_len
        self.tranco_dict = tranco_dict
        self.chars = string.ascii_lowercase + string.digits + ".,:;/?=&-_@%#+()[]~"
        self.char2idx = {c: i + 2 for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars) + 2

    def normalize_url(self, url):
        url = str(url).lower().strip()
        url = re.sub(r'^https?://', '', url)
        url = re.sub(r'^www\.', '', url)
        return url.rstrip('/')

    def entropy(self, s):
        if not s: return 0.0
        probs = [n / len(s) for n in Counter(s).values()]
        return -sum(p * math.log2(p) for p in probs)

    def is_ip_address(self, domain):
        return 1 if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', domain) else 0

    def get_rank_raw(self, url):
        try:
            # Using tldextract to extract the domain from the clean link
            ext = tldextract.extract(url)
            domain = f"{ext.domain}.{ext.suffix}"
            # Searching the dictionary we passed on to the class
            rank = self.tranco_dict.get(domain)
            return float(rank) if rank else 1000001.0
        except:
            return 1000001.0

    def extract_manual_features(self, clean_url, raw_url):
        parsed = urlparse(f"http://{clean_url}")
        domain = parsed.netloc
        path = parsed.path

        # Extract the rank here to be added to the list
        rank_val = self.get_rank_raw(clean_url)

        features = [
            len(clean_url),
            len(raw_url),
            self.entropy(clean_url),
            self.entropy(domain),
            sum(c.isdigit() for c in clean_url),
            sum(c in "-_@%." for c in clean_url),
            clean_url.count('.'),
            clean_url.count('/'),
            self.is_ip_address(domain),
            len(path),
            rank_val # <--- We added it here to be processed along with the rest of the numbers in StandardScaler
        ]
        return np.array(features, dtype=np.float32)

    def char_encode(self, url):
        seq = [self.char2idx.get(c, 1) for c in url]
        if len(seq) > self.max_len:
            seq = seq[:self.max_len]
        else:
            seq = seq + [0] * (self.max_len - len(seq))
        return np.array(seq, dtype=np.int32)

    def process(self, url):
        raw_url = str(url)
        clean_url = self.normalize_url(raw_url)
        seq = self.char_encode(clean_url)
        feats = self.extract_manual_features(clean_url, raw_url)

        # Domain extraction for partitioning
        domain = clean_url.split('/')[0] if '/' in clean_url else clean_url

        return seq, feats, domain

# --- Processing Application ---
preprocessor = URLPreprocessor(tranco_dict=tranco_dict, max_len=MAX_LEN)

print("Processing Dataset...")
results = df['url'].parallel_apply(preprocessor.process)

# Results Chapter
X_seq_all = np.stack(results.map(lambda x: x[0]).values)
X_feat_all = np.stack(results.map(lambda x: x[1]).values)
domains_all = results.map(lambda x: x[2]).values
y_all = df['label'].values

# Scaling
scaler = StandardScaler()
X_feat_all = scaler.fit_transform(X_feat_all)

# --- (Domain-Aware Splitting) ---
print("Splitting Data...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(X_seq_all, y_all, groups=domains_all))

# Training data (2 inputs + 1 output)
X_train_seq = X_seq_all[train_idx]
X_train_feat = X_feat_all[train_idx]
y_train = y_all[train_idx]

# Verification Data
X_val_seq = X_seq_all[val_idx]
X_val_feat = X_feat_all[val_idx]
y_val = y_all[val_idx]

print(f"Ready for Hybrid Model:")
print(f"Train Seq: {X_train_seq.shape}, Feat: {X_train_feat.shape}")
print(f"Val Seq: {X_val_seq.shape}, Feat: {X_val_feat.shape}")

# **Model Architecture**


---


# **CNN + BiGRU**

In [ ]:
# ==========================================
# 2.(Hybrid CNN-GRU + Dense)
# ==========================================
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Concatenate, Flatten

# --- First Entry: Text Sequence (for CNN/GRU) ---
input_seq = Input(shape=(MAX_LEN,), name='sequence_input')
x1 = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(input_seq)

# CNN-1 Layers (for extracting patterns from text)
x1 = Conv1D(filters=128, kernel_size=3, padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = Activation('relu')(x1)

# CNN-2 Layers
x1 = Conv1D(filters=128, kernel_size=4, padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = Activation('relu')(x1)

x1 = MaxPooling1D(pool_size=2)(x1)
x1 = Dropout(0.3)(x1)

# Bi-GRU Layer (for context understanding)
x1 = Bidirectional(GRU(units=128, return_sequences=False))(x1)

# --- Second Entry: Digital Properties (of Dense) ---
input_feat = Input(shape=(X_train_feat.shape[1],), name='features_input')
x2 = Dense(64, activation='relu')(input_feat)
x2 = BatchNormalization()(x2)
x2 = Dropout(0.2)(x2)

# --- Concatenate the two inputs ---
combined = Concatenate()([x1, x2])

# --- Final Classifiers --- # Here we place the powerful classes because the model needs to process the embedded information
z = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(combined)
z = BatchNormalization()(z)
z = Dropout(0.4)(z)

z = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(z)
z = Dropout(0.3)(z)

# Final Exit
output = Dense(1, activation='sigmoid')(z)
# Model Definition
model = Model(inputs=[input_seq, input_feat], outputs=output)

#(Compile)
optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

# **Training**

In [ ]:
# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True, verbose=1
)

checkpoint = ModelCheckpoint(
    'best_url_model.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=1, min_lr=1e-6, verbose=1
)

print("start train...")
history = model.fit(
    [X_train_seq, X_train_feat], y_train,  # validation_data=(X_val, y_val),
    validation_data=([X_val_seq, X_val_feat], y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, checkpoint, reduce_lr],
    shuffle=True
)

# **Plotting Curves**

In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.show()

plot_history(history)

# **Evaluation Model base on External test file**

In [ ]:
model=load_model('/kaggle/working/final_url_model.keras')

In [ ]:
if os.path.exists(TEST_PATH):
    df_test = pd.read_csv(TEST_PATH)

    # clean
    df_test.columns = df_test.columns.str.strip().str.lower()
    df_test.dropna(subset=['url', 'label'], inplace=True)

    # convarte label
    df_test['label'] = df_test['label'].replace(mapping)
    df_test['label'] = pd.to_numeric(df_test['label'], errors='coerce')
    df_test.dropna(subset=['label'], inplace=True)

    # --- (Seq + Features) ---
    print("Processing Test Data...")
    results_test = df_test['url'].parallel_apply(preprocessor.process)

    X_test_seq = np.stack(results_test.map(lambda x: x[0]).values)
    X_test_feat = np.stack(results_test.map(lambda x: x[1]).values)
    y_test_true = df_test['label'].astype(int).values

    # Scaling
    X_test_feat = scaler.transform(X_test_feat)

    # Prediction
    loss, accuracy = model.evaluate([X_test_seq, X_test_feat], y_test_true, verbose=1)
    print(f"\nTest Accuracy: {accuracy*100:.2f}%")

    # Report
    y_pred_probs = model.predict([X_test_seq, X_test_feat])
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()

    print("\n--- Classification Report ---")
    print(classification_report(y_test_true, y_pred, target_names=['Benign', 'Malicious']))

    # Confusion Matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_test_true, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - Test Data')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

else:
    print(f"test file it's not exist {TEST_PATH}")

# **Evaluation Metrics for Model**

In [ ]:
print("📈...")
precision = precision_score(y_test_true, y_pred)
recall = recall_score(y_test_true, y_pred)
f1 = f1_score(y_test_true, y_pred)

# ROC-AUC
roc_auc = roc_auc_score(y_test_true, y_pred_probs)

print("-" * 30)
print(f"Precision: {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("-" * 30)


plt.figure(figsize=(14, 6))

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test_true, y_pred_probs)
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# --- Precision-Recall Curve ---
# (Imbalanced Data)
precision_curve, recall_curve, _ = precision_recall_curve(y_test_true, y_pred_probs)
pr_auc = auc(recall_curve, precision_curve)

plt.subplot(1, 2, 2)
plt.plot(recall_curve, precision_curve, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# **Saving Assets**

In [ ]:
import pickle

print("Saving Assets...")

# 1. Save Model
model.save('final_url_model.keras')

# 2. Save Scaler
with open('scaler.pickle', 'wb') as handle:
    pickle.dump(scaler, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open('preprocessor.pickle', 'wb') as handle:
#     pickle.dump(preprocessor, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model & Scaler saved successfully")

# **Inference System**

In [ ]:
def predict_custom_url(url_string):
    # 1. Preprocessing using our custom class
    # return function (Sequence, Features, Domain)
    seq, feat, _ = preprocessor.process(url_string)

    # 2. Reshape inputs to match model expectation (Batch size of 1)

    # 1. Adjusting text dimensions
    seq = seq.reshape(1, -1)

    # 2. Adjust the feature dimensions to be (1, 11)
    fet = fet.reshape(1, -1)

    fet_old = fet[:, :10]   # First 10 columns
    fet_new = fet[:, 10:]   # Column No. 11 (Rank)

    #4. Applying the scaler to only the 10 features
    fet_old_scaled = scaler.transform(fet_old)

    # (Scaled 10 features + Raw Rank feature)
    final_features = np.hstack((fet_old_scaled, fet_new))

    # 6.Predict
    score = model.predict([seq, final_features], verbose=0)[0][0]
    label = "Malicious" if score > 0.5 else "Benign"
    confidence = score if score > 0.5 else 1 - score

    return label, confidence

print("Enter 'exit' to quit")

while True:
    user_url = input("\nEnter URL: ")
    if user_url.lower() in ['exit', 'quit']:
        print("--👋--")
        break

    if user_url.strip():
        status, conf = predict_custom_url(user_url)
        print(f"Result : {status} (confidence: {conf*100:.2f}%)")
    else:
        print("Enter a valid URL")